In [11]:
from committor_control import *

import pandas as pd
import numpy as np
import os, sys, re
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path

plt.rcParams['xtick.labelsize'] = 14  # font size
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams["font.family"] = "serif"
mpl.rcParams['pdf.fonttype'] = 42  # Use TrueType fonts (editable in Illustrator)
mpl.rcParams['ps.fonttype'] = 42   # Same for EPS
mpl.rcParams['svg.fonttype'] = 'none'  # Keep text as text in SVG
mpl.rcParams["mathtext.fontset"] =  "cm"

In [ ]:
if __name__ == "__main__":
    dt = 0.01
    num_trajs = 1000
 
    sys_dw   = make_double_well_curved(lambda_u=2.0, lambda_s=8.0, c=0.5)
    sys_lin  = make_linear_system(lambda_u=2.0, lambda_s=-8.0)
    sys_pend = make_pendulum(g=1.5, length=1.0, damping=0.0)
 
    use_whitened = True
 
    systems = {
        "linear_2d":   (sys_lin,  1.0, np.inf, 10, 0.01),
        "double_well": (sys_dw,   0.5, 0.5,    10, 0.01),
        "pendulum":    (sys_pend, 1.0, 1.0,    30, 0.01),
    }
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 1: Trajectory examples (all 3 systems)
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 1: Trajectory examples\n" + "#"*80)
    traj_methods = ["Autonomous", "CCA (Whitened)", "CCA (Adaptive)"]
    for tag, (sys, a, r_neigh, tau, D_eval) in systems.items():
        run_evaluation_suite(sys, a=a, r_neigh=r_neigh, D_eval=D_eval, dt=dt,
                             num_trajs=100, plot_trajs=3, spread_s=0.75, tau=tau,
                             methods_to_run=traj_methods, use_whitened=use_whitened,
                             file_suffix="trajs")
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 2: Linear system diagnostics
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 2: Linear diagnostics\n" + "#"*80)
    sys, a, r_neigh, tau, D_eval = systems["linear_2d"]
 
    # 2a. Convergence
    conv_csv = "figures/linear_2d_convergence.csv"
    generate_convergence_data(sys, a=a, D_train=D_eval, dt=dt, T_train=500,
                              r_neigh=r_neigh, num_trajs=num_trajs, tau=tau,
                              use_whitened=use_whitened, csv_filepath=conv_csv)
    # FIX: removed legend_fontsize (not in doc 9's signature)
    plot_convergence_from_data(csv_filepath=conv_csv, sys_lambda_true=sys.lambda_u_true,
                               sys_D_true=D_eval, filepath="figures/linear_2d_convergence.pdf",
                               layout="stacked", label_fontsize=13, tick_fontsize=12)
 
    # 2b. Noise sweep — FIX: use plot_scaling_and_energy (not plot_success_and_energy_vs_noise)
    energy_csv = "figures/linear_2d_noise_sweep.csv"
    simulate_and_save_noise_energy_sweep(sys, a=a, dt=dt, r_neigh=r_neigh,
                                         num_trajs=num_trajs, T_sim=2000, tau=tau,
                                         use_whitened=use_whitened, csv_filepath=energy_csv)
    plot_scaling_and_energy(energy_csv, sys_name=sys.name,
                            filepath="figures/linear_2d_noise_sweep.pdf",
                            mode="stacked", include_adaptive=False)
 
    # 2c. Dimensionality & lambda scaling
    run_dimensionality_scaling_experiment(dt=dt, num_trajs=num_trajs, T_sim=5000, use_whitened=use_whitened)
 
    # 2d. Combined scaling + energy — FIX: use plot_combined_scaling_and_energy
    plot_combined_scaling_and_energy(
        dim_csv_filepath="figures/dimensionality_scaling_stats_WHITENED_CCA.csv",
        energy_csv_filepath=energy_csv,
        include_adaptive=False,
        filepath="figures/linear_2d_combined_scaling_energy.pdf")
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 3: Nonlinear systems — noise sweep
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 3: Nonlinear noise sweeps\n" + "#"*80)
    for tag in ["double_well", "pendulum"]:
        sys, a, r_neigh, tau, D_eval = systems[tag]
        csv_path = f"figures/{tag}_noise_sweep.csv"
        simulate_and_save_noise_energy_sweep(sys, a=a, dt=dt, r_neigh=r_neigh,
                                              num_trajs=num_trajs, T_sim=2000, tau=tau,
                                              use_whitened=use_whitened, csv_filepath=csv_path)
        # FIX: use plot_scaling_and_energy (not plot_success_and_energy_vs_noise)
        plot_scaling_and_energy(csv_path, sys_name=sys.name,
                                filepath=f"figures/{tag}_noise_sweep.pdf",
                                mode="stacked", include_adaptive=False)
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 4: Baseline comparisons
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 4: Baseline comparisons\n" + "#"*80)
    for tag in ["linear_2d", "double_well"]:
        sys, a, r_neigh, tau, D_eval = systems[tag]
        csv_path = f"figures/{tag}_baselines.csv"
        run_baseline_comparison(sys, a=a, r_neigh=r_neigh, D_eval=D_eval, dt=dt,
                                num_trajs=500, T_sim=2000, T_train=2500, tau=tau,
                                csv_filepath=csv_path)
        # FIX: removed sys.name positional arg (function doesn't take it)
        plot_baseline_comparison(csv_filepath=csv_path,
                                 filepath=f"figures/{tag}_baselines.pdf")
 
    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 5: Ablations
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 5: Ablations\n" + "#"*80)
 
    # 5a. Trust-region (Neighborhood) Ablation - Reproduces Table I
    for tag in ["double_well", "pendulum"]:
        sys, a, r_neigh, tau, D_eval = systems[tag]
        csv_path = f"figures/{tag}_trust_ablation.csv"
        
        # We use the specific levels from Table I to show the trade-off
        # between data starvation (0.1) and nonlinearity poisoning (inf)
        run_trust_region_ablation(sys, a=a, D_eval=D_eval, dt=dt, num_trajs=500,
                                  T_sim=2000, T_train=2500, tau=tau,
                                  r_neigh_values=[0.1, 0.25, 0.5, 1.0, np.inf], 
                                  csv_filepath=csv_path)
        
        plot_trust_region_ablation(csv_path, sys.name, f"figures/{tag}_trust_ablation.pdf")
 
    # 5b. Estimator Robustness (Lag & Discount Factor)
    sys, a, r_neigh, tau, D_eval = systems["linear_2d"]
    csv_path = "figures/linear_2d_hyperparam_ablation.csv"
    run_hyperparameter_ablation(sys, a=a, r_neigh=r_neigh, D_eval=D_eval, dt=dt,
                                 num_trajs=500, T_sim=2000, T_train=2500,
                                 tau_values=[1, 5, 10, 20, 50],
                                 gamma_values=[0.0, 0.005, 0.01, 0.05],
                                 csv_filepath=csv_path)
    
    plot_hyperparameter_ablation(csv_path, sys.name, "figures/linear_2d_hyperparam_ablation.pdf",
                                 true_lambda=sys.lambda_u_true)

In [ ]:
from committor_control import *

In [18]:
import numpy as np
import csv

def run_wsvd_robustness_sweep(dt=0.01, T_train=5000, num_trials=1000):
    """
    Measures the alignment error between \\hat{w}_u and the true w_u 
    across a range of noise levels D.
    """
    # Define a standard non-normal linear saddle
    sys = make_linear_system(lambda_u=2.0, lambda_s=-8.0)
    w_u_true = sys.w_u_true / np.linalg.norm(sys.w_u_true)
    
    # Noise levels from negligible to significant
    noise_levels = [0.0, 0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, .5, 1.]
    results = []

    print(f"{'Noise D':>10} | {'Median Error (deg)':>20} | {'Cos Similarity':>15}")
    print("-" * 52)

    for D in noise_levels:
        trial_errors = []
        for seed in range(num_trials):
            # Algorithm 1: Training phase
            # Fix: Use sys.x_star.size instead of sys.dim
            tracker = CCATracker(dim=sys.x_star.size, dt=dt, tau=20, use_whitened=True)
            tracker.train(sys.drift, T=T_train, dt=dt, D=D, seed=seed, 
                          a=1.0, sign_ref=w_u_true, init_center=sys.x_star)
            
            p = tracker.get_params(w_u_true)
            w_u_hat = p['w_u']
            
            # Metric: Absolute cosine similarity (sign-invariant)
            cos_sim = np.abs(np.dot(w_u_hat, w_u_true))
            angle_err = np.degrees(np.arccos(np.clip(cos_sim, 0, 1)))
            trial_errors.append(angle_err)
        
        median_err = np.median(trial_errors)
        results.append((D, median_err))
        print(f"{D:10.4f} | {median_err:20.2f}° | {np.cos(np.radians(median_err)):15.4f}")

    return results

# Run the sweep
if __name__ == "__main__":
    results = run_wsvd_robustness_sweep() 

   Noise D |   Median Error (deg) |  Cos Similarity
----------------------------------------------------
    0.0000 |                 0.00° |          1.0000
    0.0001 |                 1.39° |          0.9997
    0.0010 |                 4.09° |          0.9975
    0.0050 |                 4.31° |          0.9972
    0.0100 |                 4.12° |          0.9974
    0.0200 |                 3.42° |          0.9982
    0.0500 |                 2.48° |          0.9991
    0.1000 |                 2.82° |          0.9988
    0.5000 |                15.68° |          0.9628
    1.0000 |                16.08° |          0.9609


In [22]:
import numpy as np
import csv

def generate_trajectory(sys, T_steps, dt, D, a, seed):
    """Simulates autonomous dynamics with boundary resets (matching CCATracker)."""
    rng = np.random.default_rng(seed)
    n = sys.x_star.size
    X = np.zeros((T_steps, n))
    x = sys.x_star + rng.normal(0, 0.05, n)
    
    w_u_norm = sys.w_u_true / np.linalg.norm(sys.w_u_true)
    
    for t in range(T_steps):
        X[t] = x
        
        # Robustly handle different drift signatures: drift(x), drift(t, x), or drift(x, u)
        try:
            d_val = sys.drift(x)
        except TypeError:
            try:
                d_val = sys.drift(t * dt, x)  # If signature is (t, x)
            except TypeError:
                d_val = sys.drift(x, 0.0)     # If signature is (x, u)
                
        dx = d_val * dt + np.sqrt(2 * D * dt) * rng.normal(size=n)
        x = x + dx
        
        # Boundary reset logic (excursion on either side for passive data)
        if np.abs(np.dot(w_u_norm, x - sys.x_star)) > a * 2.0:
            x = sys.x_star + rng.normal(0, 0.05, n)
            
    return X

def estimate_wu_from_states(X, tau):
    """Estimates w_u using raw centered states instead of finite differences."""
    # 1. Center the states (Absolutely necessary to remove fixed point offset)
    mu = np.mean(X, axis=0)
    X_c = X - mu
    
    # 2. Create lagged pairs
    P = X_c[:-tau]
    F = X_c[tau:]
    
    # 3. Covariances
    C_pp = P.T @ P / len(P)
    C_fp = F.T @ P / len(P)
    
    # 4. Whitening and SVD
    vals, vecs = np.linalg.eigh(C_pp)
    vals = np.maximum(vals, 1e-6) # Spectral floor
    inv_sqrt = vecs @ np.diag(1.0 / np.sqrt(vals)) @ vecs.T
    
    M = inv_sqrt @ C_fp @ inv_sqrt
    
    # 5. Extract leading left singular vector
    U, S, Vh = np.linalg.svd(M)
    psi = U[:, 0]
    
    w_u_hat = inv_sqrt @ psi
    
    # Sign align (heuristic: assume positive projection on x[0] for consistency)
    if w_u_hat[0] < 0:
        w_u_hat = -w_u_hat
        
    return w_u_hat / np.linalg.norm(w_u_hat)

def run_state_based_sweep(dt=0.01, T_train=5000, num_trials=1000):
    sys = make_linear_system(lambda_u=2.0, lambda_s=-8.0)
    w_u_true = sys.w_u_true / np.linalg.norm(sys.w_u_true)
    
    noise_levels = [0.0, 0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, .5, 1.]
    results = []

    print(f"{'Noise D':>10} | {'Median Error (deg)':>20} | {'Cos Similarity':>15}")
    print("-" * 52)

    for D in noise_levels:
        trial_errors = []
        tau_steps = max(1, int(0.1 / dt)) # 0.1s physical lag
        
        for seed in range(num_trials):
            # 1. Generate Raw Data
            X = generate_trajectory(sys, T_train, dt, D, a=1.0, seed=seed)
            
            # 2. State-Based Estimation
            w_u_hat = estimate_wu_from_states(X, tau=tau_steps)
            
            # Metric: Absolute cosine similarity
            cos_sim = np.abs(np.dot(w_u_hat, w_u_true))
            angle_err = np.degrees(np.arccos(np.clip(cos_sim, 0, 1)))
            trial_errors.append(angle_err)
        
        median_err = np.median(trial_errors)
        results.append((D, median_err))
        print(f"{D:10.4f} | {median_err:20.2f}° | {np.cos(np.radians(median_err)):15.4f}")

    return results

if __name__ == "__main__":
    results = run_state_based_sweep(num_trials=100) # Reduced to 100 for speed

   Noise D |   Median Error (deg) |  Cos Similarity
----------------------------------------------------
    0.0000 |                41.01° |          0.7546
    0.0001 |                45.27° |          0.7038
    0.0010 |                38.87° |          0.7785
    0.0050 |                53.66° |          0.5926
    0.0100 |                42.94° |          0.7321
    0.0200 |                56.04° |          0.5586
    0.0500 |                61.10° |          0.4833
    0.1000 |                61.60° |          0.4756
    0.5000 |                72.95° |          0.2932
    1.0000 |                77.26° |          0.2205


In [20]:
import os
import numpy as np
from functools import partial
import matplotlib.pyplot as plt

# (Assume SystemSpec, make_linear_system, CCATracker, erf_feedback, 
# and simulate_trajectory from the main codebase are already imported here)

def run_dt_bias_experiment():
    lam_u, lam_s, D, a = 2.0, -8.0, 0.01, 1.0
    sys = make_linear_system(lambda_u=lam_u, lambda_s=lam_s)
    
    print(f"System: lambda_u={lam_u}, lambda_s={lam_s}, D={D}, a={a}")
    print(f"w_u = {sys.w_u_true}\n")
    
    dt_values = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
    n_traj = 500
    n_est = 30          # number of independent passive trajectories for estimation
    T_sim = 10.0
    T_passive = 5.0
    
    keys = ['dt', 'auto_sr', 'oracle_sr', 'oracle_energy',
            'data_sr', 'data_energy', 'lam_bias', 'lam_std',
            'D_ratio_mean', 'D_ratio_std', 'cos_w_mean', 'cos_w_std']
    results = {k: [] for k in keys}
    
    w_u_norm = sys.w_u_true / np.linalg.norm(sys.w_u_true)
    w_s = np.array([-w_u_norm[1], w_u_norm[0]]) # Orthogonal stable direction
    
    for dt in dt_values:
        print(f"=== dt = {dt} ===")
        steps = int(T_sim / dt)
        steps_passive = int(T_passive / dt)
        
        # Keep the physical time lag constant at 0.1 seconds for fair CCA comparison
        physical_lag = 0.1
        tau_steps = max(1, int(physical_lag / dt))
        
        # Generate Initial Conditions (Undesired side)
        rng_ic = np.random.default_rng(42)
        ics = [sys.x_star + rng_ic.uniform(0, 0.75 * a) * w_u_norm + 
               rng_ic.uniform(-0.75 * a, 0.75 * a) * w_s for _ in range(n_traj)]
        
        # Helper to run batches
        def run_batch(ctrl, seed):
            batch_rng = np.random.default_rng(seed)
            A_list, E_list = [], []
            for x0 in ics:
                _, hit, energy = simulate_trajectory(
                    x0, steps, dt, sys.drift, D, ctrl, sys, a, batch_rng
                )
                A_list.append(1.0 if hit == "reached_A" else 0.0)
                E_list.append(energy)
            return np.array(A_list), np.array(E_list)
        
        # 1. Autonomous
        auto_A, _ = run_batch(ctrl=None, seed=100)
        
        # 2. Oracle
        c_oracle = partial(erf_feedback, w_u=sys.w_u_true, lambda_u=lam_u, D=D, x_fp=sys.x_star, a=a)
        oracle_A, oracle_E = run_batch(ctrl=c_oracle, seed=200)
        
        # 3. Data-enabled: estimate from multiple passive trajectories
        lam_ests, D_ests, cos_ests, w_hats, fp_hats = [], [], [], [], []
        
        for k in range(n_est):
            tracker = CCATracker(dim=sys.x_star.size, dt=dt, tau=tau_steps, use_whitened=True)
            tracker.train(sys.drift, T=steps_passive, dt=dt, D=D, seed=5000+k, 
                          a=a, sign_ref=sys.w_u_true, init_center=sys.x_star)
            
            p = tracker.get_params(sys.w_u_true)
            w_hat = p['w_u']
            lam_hat = p['lambda_u']
            D_hat = p['D']
            fp_hat = p['x_star']
            
            # Filter out severely failed estimates (e.g. if trajectory blew up immediately)
            if lam_hat > 0:
                lam_ests.append(lam_hat)
                D_ests.append(D_hat)
                cos_ests.append(np.abs(np.dot(w_hat, sys.w_u_true)))
                w_hats.append(w_hat)
                fp_hats.append(fp_hat)
        
        lam_arr = np.array(lam_ests) if lam_ests else np.array([np.nan])
        D_arr = np.array(D_ests) if D_ests else np.array([np.nan])
        cos_arr = np.array(cos_ests) if cos_ests else np.array([np.nan])
        
        # Use median estimates, pick w_hat with best cos
        if len(w_hats) > 0:
            best_k = np.argmax(cos_arr)
            w_use = w_hats[best_k]
            lam_use = np.median(lam_arr)
            D_use = np.median(D_arr)
            fp_use = fp_hats[best_k]
            
            c_data = partial(erf_feedback, w_u=w_use, lambda_u=lam_use, D=D_use, x_fp=fp_use, a=a)
            data_A, data_E = run_batch(ctrl=c_data, seed=200)
            
            # Compute conditional mean energy (only successful trajectories)
            succ_mask = data_A == 1.0
            data_sr = np.mean(data_A)
            data_en = np.mean(data_E[succ_mask]) if np.any(succ_mask) else 0.0
        else:
            data_sr = 0.0
            data_en = 0.0
        
        # Compute conditional mean oracle energy
        o_succ_mask = oracle_A == 1.0
        oracle_en = np.mean(oracle_E[o_succ_mask]) if np.any(o_succ_mask) else 0.0
        
        results['dt'].append(dt)
        results['auto_sr'].append(np.mean(auto_A))
        results['oracle_sr'].append(np.mean(oracle_A))
        results['oracle_energy'].append(oracle_en)
        results['data_sr'].append(data_sr)
        results['data_energy'].append(data_en)
        results['lam_bias'].append(np.nanmean(lam_arr) - lam_u)
        results['lam_std'].append(np.nanstd(lam_arr))
        results['D_ratio_mean'].append(np.nanmean(D_arr / D))
        results['D_ratio_std'].append(np.nanstd(D_arr / D))
        results['cos_w_mean'].append(np.nanmean(cos_arr))
        results['cos_w_std'].append(np.nanstd(cos_arr))
        
        print(f"  Auto SR:   {np.mean(auto_A):.3f}")
        print(f"  Oracle SR: {np.mean(oracle_A):.3f}, E={oracle_en:.4f}")
        print(f"  Data SR:   {data_sr:.3f}, E={data_en:.4f}")
        if len(lam_ests) > 0:
            print(f"  lam_hat = {np.mean(lam_arr):.3f} +/- {np.std(lam_arr):.3f}")
            print(f"  D_hat/D = {np.mean(D_arr/D):.3f} +/- {np.std(D_arr/D):.3f}")
            print(f"  |cos|   = {np.mean(cos_arr):.4f} +/- {np.std(cos_arr):.4f}")
        print()
    
    return results

def plot_results(results):
    dt = np.array(results['dt'])
    c_or, c_da = '#1565C0', '#E65100'
    
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    fig.suptitle(r'Discrete-Time Bias: $\lambda_u\!=\!2$, $D\!=\!0.01$, $a\!=\!1$', fontsize=13, fontweight='bold')
    
    # (A) Success rate
    ax = axes[0, 0]
    ax.semilogx(dt, results['oracle_sr'], 'o-', color=c_or, label='Oracle', lw=2)
    ax.semilogx(dt, results['data_sr'], 's--', color=c_da, label='Data-enabled', lw=2)
    ax.semilogx(dt, results['auto_sr'], 'x:', color='gray', label='Autonomous', lw=1.5)
    ax.set_xlabel(r'$\Delta t$')
    ax.set_ylabel('Success rate')
    ax.set_title('(A) Success rate')
    ax.legend(fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    
    # (B) Oracle energy vs dt
    ax = axes[0, 1]
    oracle_e = np.array(results['oracle_energy'])
    ax.semilogx(dt, oracle_e, 'o-', color=c_or, lw=2, label='Oracle energy')
    coeffs = np.polyfit(dt, oracle_e, 1)
    dt_fine = np.linspace(dt[0], dt[-1], 100)
    ax.semilogx(dt_fine, np.polyval(coeffs, dt_fine), ':', color=c_or, alpha=0.5, label=rf'Linear fit (slope={coeffs[0]:.2f})')
    ax.set_xlabel(r'$\Delta t$')
    ax.set_ylabel('Mean control energy (Succ)')
    ax.set_title(r'(B) Oracle energy vs $\Delta t$')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # (C) lam_bias
    ax = axes[1, 0]
    bias = np.array(results['lam_bias'])
    std = np.array(results['lam_std'])
    ax.semilogx(dt, bias, 's-', color=c_da, lw=2)
    ax.fill_between(dt, bias - std, bias + std, color=c_da, alpha=0.15)
    ax.axhline(0, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel(r'$\Delta t$')
    ax.set_ylabel(r'$\hat{\lambda}_u - \lambda_u$')
    ax.set_title('(C) Growth rate bias')
    ax.grid(True, alpha=0.3)
    
    # (D) D_ratio and |cos|
    ax = axes[1, 1]
    D_ratio = np.array(results['D_ratio_mean'])
    cos_m = np.array(results['cos_w_mean'])
    ax.semilogx(dt, D_ratio, 's-', color='#D32F2F', lw=2, label=r'$\hat{D}/D$')
    ax.semilogx(dt, cos_m, 'D-', color='#2E7D32', lw=2, label=r'$|\cos(\hat{w}_u, w_u)|$')
    ax.axhline(1, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel(r'$\Delta t$')
    ax.set_ylabel('Ratio / Cosine')
    ax.set_title('(D) Estimation quality')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/dt_bias_v4.png', dpi=150, bbox_inches='tight')
    plt.savefig('figures/dt_bias_v4.pdf', bbox_inches='tight')
    print("Saved plots to 'figures/' directory.")

if __name__ == '__main__':
    results = run_dt_bias_experiment()
    plot_results(results)
    
    print("\n" + "="*100)
    hdr = f"{'dt':>7} | {'Auto':>5} | {'Orac':>5} | {'Data':>5} | {'Or.E':>7} | {'Da.E':>7} | {'lam_b':>7} | {'D/D':>5} | {'|cos|':>6}"
    print(hdr)
    print("-"*100)
    for i in range(len(results['dt'])):
        print(f"{results['dt'][i]:7.3f} | "
              f"{results['auto_sr'][i]:5.3f} | "
              f"{results['oracle_sr'][i]:5.3f} | "
              f"{results['data_sr'][i]:5.3f} | "
              f"{results['oracle_energy'][i]:7.4f} | "
              f"{results['data_energy'][i]:7.4f} | "
              f"{results['lam_bias'][i]:+7.3f} | "
              f"{results['D_ratio_mean'][i]:5.2f} | "
              f"{results['cos_w_mean'][i]:6.4f}")

System: lambda_u=2.0, lambda_s=-8.0, D=0.01, a=1.0
w_u = [0.99590457 0.09041061]

=== dt = 0.001 ===
  Auto SR:   0.038
  Oracle SR: 1.000, E=0.8486
  Data SR:   1.000, E=0.8501
  lam_hat = 2.015 +/- 0.143
  D_hat/D = 1.001 +/- 0.019
  |cos|   = 0.9614 +/- 0.0293

=== dt = 0.002 ===
  Auto SR:   0.050
  Oracle SR: 1.000, E=0.8485
  Data SR:   1.000, E=0.8337
  lam_hat = 1.997 +/- 0.201
  D_hat/D = 0.993 +/- 0.022
  |cos|   = 0.9587 +/- 0.0433

=== dt = 0.005 ===
  Auto SR:   0.050
  Oracle SR: 1.000, E=0.8370
  Data SR:   1.000, E=0.8893
  lam_hat = 1.935 +/- 0.157
  D_hat/D = 0.988 +/- 0.051
  |cos|   = 0.9632 +/- 0.0327

=== dt = 0.01 ===
  Auto SR:   0.046
  Oracle SR: 1.000, E=0.8614
  Data SR:   1.000, E=0.8557
  lam_hat = 1.926 +/- 0.171
  D_hat/D = 0.994 +/- 0.068
  |cos|   = 0.9757 +/- 0.0239

=== dt = 0.02 ===
  Auto SR:   0.034
  Oracle SR: 1.000, E=0.8516
  Data SR:   1.000, E=0.8830
  lam_hat = 1.985 +/- 0.165
  D_hat/D = 1.009 +/- 0.089
  |cos|   = 0.9716 +/- 0.0370

=== d

'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
meta NOT subset; don't know how to subset; dropped
meta NOT subset; don't know how to subset; dropped


Saved plots to 'figures/' directory.

     dt |  Auto |  Orac |  Data |    Or.E |    Da.E |   lam_b |   D/D |  |cos|
----------------------------------------------------------------------------------------------------
  0.001 | 0.038 | 1.000 | 1.000 |  0.8486 |  0.8501 |  +0.015 |  1.00 | 0.9614
  0.002 | 0.050 | 1.000 | 1.000 |  0.8485 |  0.8337 |  -0.003 |  0.99 | 0.9587
  0.005 | 0.050 | 1.000 | 1.000 |  0.8370 |  0.8893 |  -0.065 |  0.99 | 0.9632
  0.010 | 0.046 | 1.000 | 1.000 |  0.8614 |  0.8557 |  -0.074 |  0.99 | 0.9757
  0.020 | 0.034 | 1.000 | 1.000 |  0.8516 |  0.8830 |  -0.015 |  1.01 | 0.9716
  0.050 | 0.044 | 1.000 | 1.000 |  0.8807 |  0.8981 |  -0.060 |  1.06 | 0.9699


In [30]:
def run_extended_baseline_comparison(sys, a=0.5, r_neigh=np.inf, D_eval=0.01, dt=0.01,
                                     num_trajs=500, T_sim=2000, T_train=2500, tau=10,
                                     noise_levels=None, csv_filepath="figures/extended_baselines.csv"):
    """
    Runs the full battery of Oracle vs. Proposed vs. Baselines vs. Ablations.
    Saves to a distinct file to avoid overwriting standard baseline results.
    """
    if noise_levels is None: 
        noise_levels = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.5, 1.0]
    
    w_u_norm = sys.w_u_true / np.linalg.norm(sys.w_u_true)
    rng_gen = np.random.default_rng(42)
    w_s = rng_gen.standard_normal(sys.x_star.size)
    w_s -= np.dot(w_s, w_u_norm) * w_u_norm
    w_s /= np.linalg.norm(w_s) + 1e-15

    os.makedirs(os.path.dirname(csv_filepath), exist_ok=True)
    print(f"\n[Extended Suite] Running: {sys.name}")

    with open(csv_filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["System", "Method", "Noise_D", "Success_Rate", "N_Total", "Mean_Energy", "Std_Energy", "Ang_Err"])

        for D_test in noise_levels:
            D_train = max(D_test, 0.005)
            kw = dict(a=a, r_neigh=r_neigh)

            # --- TRAIN ESTIMATORS ---
            # Algorithm 1 (The Full Proposed Stack)
            alg_cca = CCATracker(dim=sys.x_star.size, dt=dt, tau=tau, use_whitened=True)
            alg_cca.train(sys.drift, T_train, dt, D_train, 0, a, sys.w_u_true, init_center=sys.x_star, r_neigh=r_neigh)
            p_cca = alg_cca.get_params(sys.w_u_true)

            # Direct VAR (Standard State-based ID)
            alg_var = DirectVARTracker(dim=sys.x_star.size, dt=dt)
            alg_var.train(sys.drift, T_train, dt, D_train, 0, a, sys.w_u_true, init_center=sys.x_star, r_neigh=r_neigh)
            p_var = alg_var.get_params(sys.w_u_true)

            # --- DEFINE CONTROLLERS ---
            def get_fn(D): return erf_feedback if D >= 1e-8 else relu_feedback

            methods = [
                ("Oracle Committor", partial(get_fn(D_test), w_u=sys.w_u_true, lambda_u=sys.lambda_u_true, D=D_test, x_fp=sys.x_star, **kw)),
                ("Oracle LQR",       partial(lqr_feedback, w_u=sys.w_u_true, lambda_u=sys.lambda_u_true, D=D_test, x_fp=sys.x_star, **kw)),
                ("Est CCA Committor", partial(get_fn(p_cca['D']), w_u=p_cca['w_u'], lambda_u=p_cca['lambda_u'], D=p_cca['D'], x_fp=p_cca['x_star'], **kw)),
                ("Est VAR Committor", partial(get_fn(p_var['D']), w_u=p_var['w_u'], lambda_u=p_var['lambda_u'], D=p_var['D'], x_fp=p_var['x_star'], **kw)),
                ("Ablation OLS",      partial(get_fn(p_cca['D']), w_u=p_cca['w_u'], lambda_u=p_cca['lambda_u']*0.75, D=p_cca['D'], x_fp=p_cca['x_star'], **kw))
            ]

            # --- EVALUATE ---
            rng_ic = np.random.default_rng(11)
            ics = [sys.x_star + rng_ic.uniform(0, a * 0.75) * w_u_norm + rng_ic.uniform(-a * 0.75, a * 0.75) * w_s for _ in range(num_trajs)]

            for method_name, ctrl in methods:
                successes, energies_succ = 0, []
                
                # Assign AngErr based on which estimator this method uses
                if "Oracle" in method_name: ang_err = 0.0
                elif "VAR" in method_name: ang_err = np.degrees(np.arccos(np.abs(np.dot(p_var['w_u'], w_u_norm))))
                else: ang_err = np.degrees(np.arccos(np.abs(np.dot(p_cca['w_u'], w_u_norm))))

                for x0 in ics:
                    run_rng = np.random.default_rng(int(rng_ic.integers(0, 10**6)))
                    _, hit_side, energy = simulate_trajectory(x0, T_sim, dt, sys.drift, D_test, ctrl, sys, a, run_rng)
                    if hit_side == "reached_A":
                        successes += 1
                        energies_succ.append(energy)
                
                sr = successes / num_trajs
                me_s = np.mean(energies_succ) if energies_succ else 0.0
                se_s = np.std(energies_succ) if energies_succ else 0.0
                
                writer.writerow([sys.name, method_name, D_test, sr, num_trajs, me_s, se_s, ang_err])
                print(f"    D={D_test:.4f} | {method_name:<20} | SR={sr:.3f} | AngErr={ang_err:.1f}°")

    print(f"Done. Extended results saved to -> {csv_filepath}")

In [31]:
def plot_extended_comparison(csv_filepath, filepath="figures/extended_baseline_plot.pdf"):
    """
    Plots a 3-panel summary: Success Rate, Control Energy, and Angular Error.
    """
    import pandas as pd
    df = pd.read_csv(csv_filepath)
    
    methods = df['Method'].unique()
    noises = sorted(df['Noise_D'].unique())
    
    # Mapping for pseudo-log axis for D=0
    non_zero = [n for n in noises if n > 0]
    pseudo_zero = min(non_zero)/3.0 if non_zero else 0.001
    df['x_plot'] = df['Noise_D'].apply(lambda x: pseudo_zero if x == 0 else x)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    colors = {
        "Oracle Committor": "green", "Oracle LQR": "black",
        "Est CCA Committor": "#1565C0", "Est VAR Committor": "#D32F2F",
        "Ablation OLS": "#E65100"
    }

    for method in methods:
        sub = df[df['Method'] == method]
        c = colors.get(method, "gray")
        ls = "--" if "Oracle" in method or "Ablation" in method else "-"
        m = "D" if "Oracle" in method else "o"

        # 1. Success Rate
        axes[0].plot(sub['x_plot'], sub['Success_Rate'], marker=m, ls=ls, color=c, label=method)
        
        # 2. Control Energy
        axes[1].plot(sub['x_plot'], sub['Mean_Energy'], marker=m, ls=ls, color=c)
        
        # 3. Angular Error
        axes[2].plot(sub['x_plot'], sub['Ang_Err'], marker=m, ls=ls, color=c)

    # Formatting
    titles = ["(A) Success Rate", "(B) Control Energy", "(C) Angular Error (deg)"]
    for i, ax in enumerate(axes):
        ax.set_xscale('log')
        ax.set_title(titles[i], fontsize=12, fontweight='bold')
        ax.set_xlabel("Noise Strength D")
        ax.grid(True, alpha=0.3)
        ax.set_xticks([pseudo_zero] + non_zero)
        ax.set_xticklabels(['0'] + [f"{n:g}" for n in non_zero], rotation=45)

    axes[0].set_ylim(-0.05, 1.05)
    axes[0].legend(fontsize=8, loc='lower left')
    
    plt.tight_layout()
    plt.savefig(filepath)
    print(f"Saved Extended Plot -> {filepath}")

In [32]:
for tag in ["linear_2d"]:
    sys, a, r_neigh, tau, D_eval = systems[tag]
    csv_path = f"figures/{tag}_extended_baselines.csv"
    run_extended_baseline_comparison(sys, a=a, r_neigh=r_neigh, D_eval=D_eval, dt=dt,
                            num_trajs=500, T_sim=2000, T_train=2500, tau=tau,
                            csv_filepath=csv_path)
    plot_extended_comparison(csv_filepath=csv_path,
                                filepath=f"figures/{tag}_extended_baselines.pdf")


[Extended Suite] Running: Linear_2D
    D=0.0000 | Oracle Committor     | SR=0.000 | AngErr=0.0°
    D=0.0000 | Oracle LQR           | SR=0.000 | AngErr=0.0°
    D=0.0000 | Est CCA Committor    | SR=0.998 | AngErr=36.2°
    D=0.0000 | Est VAR Committor    | SR=1.000 | AngErr=2.6°
    D=0.0000 | Ablation OLS         | SR=0.998 | AngErr=36.2°
    D=0.0010 | Oracle Committor     | SR=1.000 | AngErr=0.0°
    D=0.0010 | Oracle LQR           | SR=0.000 | AngErr=0.0°
    D=0.0010 | Est CCA Committor    | SR=0.998 | AngErr=36.2°
    D=0.0010 | Est VAR Committor    | SR=1.000 | AngErr=2.6°
    D=0.0010 | Ablation OLS         | SR=0.998 | AngErr=36.2°
    D=0.0050 | Oracle Committor     | SR=1.000 | AngErr=0.0°
    D=0.0050 | Oracle LQR           | SR=0.000 | AngErr=0.0°
    D=0.0050 | Est CCA Committor    | SR=0.998 | AngErr=36.2°
    D=0.0050 | Est VAR Committor    | SR=1.000 | AngErr=2.6°
    D=0.0050 | Ablation OLS         | SR=0.998 | AngErr=36.2°
    D=0.0100 | Oracle Committor     | SR=1

ValueError: Multi-dimensional indexing (e.g. `obj[:, None]`) is no longer supported. Convert to a numpy array before indexing instead.